# Stage 1 — Data Ingestion & Sensor Alignment

This notebook walks through every step of `pipeline/ingest.py` interactively so
you can inspect intermediate results and confirm coverage before saving
`data/aligned.parquet`.

**Run order**: execute all cells top-to-bottom.  The final cell runs the full
pipeline and prints the quality report.  Read the report before proceeding to
`02_sampling.ipynb`.

### Potential stop conditions
| Code | Condition | Action |
|------|-----------|--------|
| STOP-1 | Match rate < 5 % | Sensor coverage gap — review Cell 6 map |
| STOP-2 | < 15 species retained | Relax `MIN_PRESENCE_COUNT` or expand radius |

In [ ]:
import sys
from pathlib import Path

# Ensure project root is importable regardless of working directory
_here = Path.cwd()
_root = _here.parent if _here.name == 'notebooks' else _here
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)-8s %(name)s  %(message)s',
    force=True,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from pipeline.ingest import (
    load_heat_map,
    load_inaturalist,
    filter_bbox,
    match_observations_to_sensors,
    filter_min_presence,
    build_aligned_dataframe,
    data_quality_report,
    run_ingest,
)
from config import (
    HEAT_MAP_PATH, INAT_PATH, ALIGNED_PATH,
    SD_LAT_MIN, SD_LAT_MAX, SD_LON_MIN, SD_LON_MAX,
    MAX_DIST_M, MAX_TIME_HRS, MIN_PRESENCE_COUNT,
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110})
print('Setup complete.  Project root:', _root)

## 1. Load raw data

In [ ]:
sensor_df = load_heat_map(HEAT_MAP_PATH)
inat_raw  = load_inaturalist(INAT_PATH)

print(f'Sensor readings      : {len(sensor_df):,}')
print(f'iNat observations    : {len(inat_raw):,}  (research-grade with timestamps)')
print()
print(f'Sensor timestamp range : {sensor_df["timestamp"].min()} → {sensor_df["timestamp"].max()}')
print(f'iNat timestamp range   : {inat_raw["time_observed_at"].min()} → {inat_raw["time_observed_at"].max()}')

In [ ]:
print('=== Sensor DataFrame ===')
display(sensor_df.head(3))
display(sensor_df.describe())

print('\n=== iNaturalist DataFrame ===')
display(inat_raw.head(3))
display(inat_raw[['latitude','longitude','taxon_name','time_observed_at']].describe())

## 2. Sensor coverage analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Unique sensor positions
sensor_locs = sensor_df[['lat','lon']].drop_duplicates()
axes[0].scatter(sensor_locs['lon'], sensor_locs['lat'], s=30, c='steelblue', alpha=0.7)
axes[0].set_title(f'Sensor locations (n={len(sensor_locs):,} unique)')
axes[0].set_xlabel('Longitude');  axes[0].set_ylabel('Latitude')

# Temperature distribution
sensor_df['temperature_c'].hist(ax=axes[1], bins=50, color='steelblue', edgecolor='none')
axes[1].set_title('Temperature distribution')
axes[1].set_xlabel('Temperature (°C)')

# Temporal density of sensor readings
sensor_df['timestamp'].dt.floor('D').value_counts().sort_index().plot(ax=axes[2], color='steelblue')
axes[2].set_title('Sensor readings per day')
axes[2].set_xlabel('Date')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 3. iNaturalist quality and species distribution

In [ ]:
# Load raw (unfiltered) iNat to show quality grade breakdown
inat_all_grades = pd.read_csv(INAT_PATH, usecols=['quality_grade'])
grade_counts = inat_all_grades['quality_grade'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

grade_counts.plot(kind='bar', ax=axes[0], color=['#2196F3','#FF9800','#9E9E9E'][:len(grade_counts)])
axes[0].set_title('iNat quality grade distribution (all grades)')
axes[0].set_xlabel('Quality grade');  axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for bar, count in zip(axes[0].patches, grade_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{count:,}', ha='center', va='bottom', fontsize=9)

# Top-20 species in research-grade data
top20 = inat_raw['taxon_name'].value_counts().head(20)
top20.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 20 species (research-grade, pre-bbox)')
axes[1].set_xlabel('Observations')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 4. Bounding-box filter

In [ ]:
inat_bbox, n_dropped_bbox = filter_bbox(inat_raw)

print(f'Observations in San Diego bbox : {len(inat_bbox):,}')
print(f'Dropped (outside bbox)         : {n_dropped_bbox:,}')

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(inat_bbox['longitude'], inat_bbox['latitude'],
           s=2, alpha=0.3, c='green', label=f'iNat in bbox (n={len(inat_bbox):,})')
rect = mpatches.Rectangle(
    (SD_LON_MIN, SD_LAT_MIN), SD_LON_MAX - SD_LON_MIN, SD_LAT_MAX - SD_LAT_MIN,
    linewidth=1.5, edgecolor='gray', facecolor='none', linestyle='--',
)
ax.add_patch(rect)
ax.set_title('iNaturalist observations in San Diego bounding box')
ax.set_xlabel('Longitude');  ax.set_ylabel('Latitude')
ax.legend()
plt.tight_layout()
plt.show()

## 5. ⚠ Coverage overlap — read this carefully

The plot below overlays the UCSD campus sensor locations (red triangles) on top
of the iNaturalist observations (green dots).  

**If the sensors form a tight cluster far from most observations, the match rate
will be very low (STOP-1 condition).**  The matching radius is 500 m; observations
more than 500 m from every sensor are automatically dropped.

In [ ]:
sensor_locs = sensor_df[['lat','lon']].drop_duplicates()

fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(
    inat_bbox['longitude'], inat_bbox['latitude'],
    s=3, alpha=0.25, c='#4CAF50', label=f'iNat observations (n={len(inat_bbox):,})',
)
ax.scatter(
    sensor_locs['lon'], sensor_locs['lat'],
    s=80, alpha=0.9, c='#F44336', marker='^', zorder=5,
    label=f'Sensor locations (n={len(sensor_locs):,} unique)',
)

# Draw 500 m radius circles around each unique sensor location
# 500 m in degrees (approximate): 500/111000 ≈ 0.0045°
_deg_radius = MAX_DIST_M / 111_000
for _, row in sensor_locs.iterrows():
    circle = plt.Circle(
        (row['lon'], row['lat']), _deg_radius,
        color='#F44336', fill=False, alpha=0.3, linewidth=0.6,
    )
    ax.add_patch(circle)

ax.set_title(
    'Coverage overlap: sensor 500 m capture zones vs iNat observations\n'
    'Observations outside all red circles will be unmatched (STOP-1 if this is most of them)',
    fontsize=11,
)
ax.set_xlabel('Longitude');  ax.set_ylabel('Latitude')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('\nSensor bounding box:')
print(f'  Lat: {sensor_locs["lat"].min():.4f} — {sensor_locs["lat"].max():.4f}')
print(f'  Lon: {sensor_locs["lon"].min():.4f} — {sensor_locs["lon"].max():.4f}')
print('\niNat observation bounding box (in SD):')
print(f'  Lat: {inat_bbox["latitude"].min():.4f} — {inat_bbox["latitude"].max():.4f}')
print(f'  Lon: {inat_bbox["longitude"].min():.4f} — {inat_bbox["longitude"].max():.4f}')

## 6. Sensor–observation matching

In [ ]:
matched_df, drop_stats = match_observations_to_sensors(inat_bbox, sensor_df)

print('Drop statistics:')
for reason, count in drop_stats.items():
    print(f'  {reason:50s}: {count:,}')

match_rate = drop_stats['matched'] / drop_stats['input_obs'] * 100 if drop_stats['input_obs'] else 0
print(f'\nMatch rate: {match_rate:.1f} %')

if match_rate < 5:
    print('\n⚠ STOP-1: Match rate below 5 %. Review the coverage map above.')

In [ ]:
if not matched_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    matched_df['_matched_dist_m'].hist(ax=axes[0], bins=40, color='steelblue', edgecolor='none')
    axes[0].set_title('Distribution of matched sensor distances')
    axes[0].set_xlabel('Distance (m)')
    axes[0].axvline(MAX_DIST_M, color='red', linestyle='--', label=f'Max ({MAX_DIST_M:.0f} m)')
    axes[0].legend()

    matched_df['_matched_time_diff_hrs'].hist(ax=axes[1], bins=40, color='steelblue', edgecolor='none')
    axes[1].set_title('Distribution of matched sensor time gaps')
    axes[1].set_xlabel('Time difference (hours)')
    axes[1].axvline(MAX_TIME_HRS, color='red', linestyle='--', label=f'Max ({MAX_TIME_HRS:.0f} h)')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print('No matches — skipping distance/time histograms.')

## 7. Species frequency and minimum-count filter

In [ ]:
if not matched_df.empty:
    species_counts = matched_df['taxon_name'].value_counts()

    fig, ax = plt.subplots(figsize=(max(10, len(species_counts)*0.4), 5))
    colors = ['#2196F3' if c >= MIN_PRESENCE_COUNT else '#EF9A9A'
              for c in species_counts.values]
    species_counts.plot(kind='bar', ax=ax, color=colors, width=0.8, edgecolor='none')
    ax.axhline(MIN_PRESENCE_COUNT, color='red', linestyle='--',
               label=f'Min threshold ({MIN_PRESENCE_COUNT} records)')
    ax.set_title('Matched records per species  (red bars = below threshold, will be dropped)')
    ax.set_xlabel('Species')
    ax.set_ylabel('Matched presence records')
    ax.legend()
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()

    print(f'Species above threshold : {(species_counts >= MIN_PRESENCE_COUNT).sum()}')
    print(f'Species below threshold : {(species_counts < MIN_PRESENCE_COUNT).sum()}')
else:
    print('No matched data — skipping species plot.')

In [ ]:
if not matched_df.empty:
    filtered_df, species_drop_stats = filter_min_presence(matched_df)
    aligned_df = build_aligned_dataframe(filtered_df)
    print(f'Rows after min-count filter : {len(aligned_df):,}')
    print(f'Species retained            : {aligned_df["taxon_name"].nunique()}')
    display(aligned_df.head())
else:
    aligned_df = pd.DataFrame()
    species_drop_stats = {}

## 8. Temporal and spatial distribution of aligned data

In [ ]:
if not aligned_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Day of year distribution (seasonality)
    aligned_df['day_of_year'].hist(ax=axes[0], bins=52, color='steelblue', edgecolor='none')
    axes[0].set_title('Seasonal distribution (day of year)')
    axes[0].set_xlabel('Day of year')

    # Temperature at matched observations
    aligned_df['temperature_c'].hist(ax=axes[1], bins=40, color='#FF7043', edgecolor='none')
    axes[1].set_title('Temperature at matched observations')
    axes[1].set_xlabel('Temperature (°C)')

    # Spatial map coloured by temperature
    sc = axes[2].scatter(
        aligned_df['lon'], aligned_df['lat'],
        c=aligned_df['temperature_c'], cmap='RdYlBu_r', s=5, alpha=0.5,
    )
    plt.colorbar(sc, ax=axes[2], label='Temp (°C)')
    axes[2].set_title('Matched observations (colour = temperature)')
    axes[2].set_xlabel('Longitude');  axes[2].set_ylabel('Latitude')

    plt.tight_layout()
    plt.show()
else:
    print('No aligned data — skipping distribution plots.')

## 9. Full pipeline run + quality report

This cell runs `run_ingest()` which repeats all the above steps and saves
`data/aligned.parquet`.  **Read the quality report before proceeding.**

In [ ]:
aligned = run_ingest()

## 10. Verify saved output

In [ ]:
if ALIGNED_PATH.exists():
    saved = pd.read_parquet(ALIGNED_PATH)
    print(f'aligned.parquet  shape : {saved.shape}')
    print(f'Species          count : {saved["taxon_name"].nunique()}')
    print(f'Columns                : {list(saved.columns)}')
    print()
    print('dtypes:')
    print(saved.dtypes)
    print()
    print('Null counts:')
    print(saved.isnull().sum())
    display(saved.head())
else:
    print('aligned.parquet not found — pipeline likely hit a STOP condition.')

In [ ]:
# Final species summary table
if ALIGNED_PATH.exists():
    saved = pd.read_parquet(ALIGNED_PATH)
    summary = (
        saved.groupby('taxon_name')
        .agg(
            n_records=('taxon_name', 'count'),
            temp_mean=('temperature_c', 'mean'),
            temp_std=('temperature_c', 'std'),
            humidity_mean=('humidity_pct', 'mean'),
        )
        .round(2)
        .sort_values('n_records', ascending=False)
    )
    print(f'\nFinal species summary ({len(summary)} species):')
    display(summary)